In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt


In [2]:
data = pd.read_csv('data/cleaned_data.csv')
data.head()

,product_name,brands,categories_en,labels_en,ingredients_text,allergens_en,additives_en,nutrition_grade_fr,energy_100g,fat_100g,...,cocoa_100g,carbon-footprint_100g,nutrition-score-fr_100g,nutrition-score-uk_100g,category_level_1,category_level_2,category_level_3,category_level_4,category_level_5,category_level_6
0,Banana Chips Sweetened (Whole),not mentioned,NaN,Labels are missing,"Bananas, vegetable oil (coconut oil, corn oil ...",unknown,No additives,d,2243.0,28.57,...,0.0,no information,14.0,14.0,NaN,NaN,NaN,NaN,NaN,NaN
1,Peanuts,torn & glasser,NaN,Labels are missing,"Peanuts, wheat flour, sugar, rice flour, tapio...","en:peanuts, en:wheat, en:soy",No additives,b,1941.0,17.86,...,0.0,no information,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,Organic Salted Nut Mix,grizzlies,NaN,Labels are missing,"Organic hazelnuts, organic cashews, organic wa...",unknown,No additives,d,2540.0,57.14,...,0.0,no information,12.0,12.0,NaN,NaN,NaN,NaN,NaN,NaN
3,Organic Polenta,bob's red mill,NaN,Labels are missing,Organic polenta,unknown,No additives,not given,1552.0,1.43,...,0.0,no information,not given,not given,NaN,NaN,NaN,NaN,NaN,NaN
4,Breadshop Honey Gone Nuts Granola,unfi,NaN,Labels are missing,"Rolled oats, grape concentrate, expeller press...",en:sesame,No additives,not given,1933.0,18.27,...,0.0,no information,not given,not given,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# us_data = pd.read_csv('data/us_data.csv')
# us_data.shape

(171521, 98)

In [ ]:
from modules.ingredients import clean_ingredients

data['ingredients'] = data['ingredients_text'].apply(clean_ingredients)
# us_data['ingredients'] = us_data['ingredients_text'].apply(clean_ingredients)

data.ingredients_text = data['ingredients_text'].apply(clean_ingredients)
# us_data.ingredients_text = us_data['ingredients_text'].apply(clean_ingredients)

### Pre-processing 

In [5]:
# Clean and standardize text columns
def clean_text(text):
    if isinstance(text, str):
        return text.lower().strip()
    return text

data['product_name'] = data['product_name'].apply(clean_text)
data['ingredients'] = data['ingredients'].apply(clean_text)
data['category_level_1'] = data['category_level_1'].apply(clean_text)
data['category_level_2'] = data['category_level_2'].apply(clean_text)

In [6]:
# Replace placeholders with NaN or empty lists
data['ingredients'] = data['ingredients'].replace('ingredients are missing', np.nan)
# Split columns into lists
data['ingredients'] = data['ingredients'].str.split(', ')

### Implement product matching

In [7]:
from fuzzywuzzy import process

def find_top_matches(user_input, choices, limit=5):
    matches = process.extract(user_input, choices, limit=limit)
    return matches

In [8]:
# # Example: User inputs a product name
# user_input = "dark chocolate bar"
# top_matches = find_top_matches(user_input, data['product_name'].tolist())

# print(f"Top matches for '{user_input}':")
# for match, score in top_matches:
#     print(f"- {match} (Score: {score})")

### Recommendations Exampe Usage

In [9]:
from modules.ingredient_recommendations import recommend_products
from modules.ingredient_recommendations import recommend_by_ingredients

# Example usage
user_input = "dark chocolate bar"  # User's input product name
allergens_to_avoid = ['wheat', 'peanuts']  # Allergens to avoid

recommendations = recommend_products(user_input, data, top_n=5, allergens_to_avoid=allergens_to_avoid)

# Print recommendations
print("Final recommendations:")
print(recommendations[['product_name', 'additives_en', 'allergens_en']].to_string(index=False))


Final recommendations:
                                             product_name              additives_en allergens_en
long grove confectionery co., day to dead chocolate skull                E322,E322i    soy, milk
           long grove confectionery co, spring nonpareils E102,E133,E322,E322i,E903    soy, milk
                                dark chocolate nonpareils           E322,E322i,E903    soy, milk
                                           dark chocolate                E322,E322i    soy, milk


### Radial chart

In [10]:
import pandas as pd
from modules.radar_chart import preprocess_data, create_radar_chart, create_radar_chart_with_dropdown

# Define the columns to analyze
category_col = 'category_level_1'
nutrient_cols = ['energy_100g', 'proteins_100g', 'carbohydrates_100g', 'fiber_100g', 'fat_100g']

# Preprocess the data
top_category_nutrition = preprocess_data(data, category_col, nutrient_cols, top_n=10)

# Extract categories and values
categories = top_category_nutrition[category_col]
values = top_category_nutrition[nutrient_cols]

# Create a radar chart
create_radar_chart(categories, values, title='Top 10 Primary Categories by Nutritional Facts')

# Create a radar chart with a dropdown
create_radar_chart_with_dropdown(categories, values, title='Top 10 Primary Categories by Nutritional Facts')

### Do not run the code from here

## Ingredient based recommendations using tokenization, embeddings and clustering

### Pre-processing and Vectorization

In [ ]:
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import nltk 
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('wordnet')  
nltk.download('omw-1.4')

ingredients = data.ingredients_text 

### Step 1 : Preprocessing with Lemmatization

In [ ]:
def preprocess(text):
    
    text = re.sub(r'[^\w\s]', '', text)
    
    # Tokenization
    tokens = word_tokenize(text)
    
    # Remove stop words
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    
    # Lemmatization
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    return ' '.join(tokens)

cleaned_ingredients = [preprocess(item) for item in ingredients]

# Vectorization using TF-IDF
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(cleaned_ingredients)

tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out())

print("Cleaned Ingredients:")
cleaned_ingredients

### Step 2 : Generate Embeddings with Sentence Transformers

In [ ]:
from sentence_transformers import SentenceTransformer

# Load a pre-trained Sentence Transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for the cleaned and lemmatized data
ingredient_embeddings = model.encode(cleaned_ingredients, show_progress_bar=True)

# Print the shape of the embeddings
print(f"Embeddings shape: {ingredient_embeddings.shape}")

### Step 3 : Clustering with K-Means

In [ ]:
from sklearn.cluster import KMeans

k = 10  
kmeans = KMeans(n_clusters=k, random_state=42)
cluster_labels = kmeans.fit_predict(ingredient_embeddings)

# Map cluster labels back to ingredients
clustered_ingredients = pd.DataFrame({
    'Ingredient': ingredients,
    'Cleaned_Ingredient': cleaned_ingredients,
    'Cluster': cluster_labels
})


print("\nClustered Data:")
print(clustered_ingredients)

### Step 4 : Similarity search with FAISS

In [ ]:
import faiss
import numpy as np

# Create a FAISS index
dimension = ingredient_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)  # L2 distance (Euclidean)
index.add(ingredient_embeddings)  # Add your embeddings to the index

# Find the 5 most similar ingredients for a query
query = "dark chocolate"
query_embedding = model.encode([query])

k = 5  # Number of nearest neighbors
distances, indices = index.search(query_embedding, k)

In [ ]:
# Retrieve the similar ingredients
similar_ingredients = [data.iloc[i]['ingredients_text'] for i in indices[0] if 0 <= i < len(data)]
print(f"Ingredients similar to '{query}': {similar_ingredients}")
print(f"Products with similar ingredients: {data.loc[indices[0], 'product_name']}")

### Save the embeddings

In [ ]:
import numpy as np

# Save embeddings to a file
np.save('embeddings/ingredient_embeddings.npy', ingredient_embeddings)

# Optionally, save the cleaned data and cluster labels for reference
import pandas as pd
clustered_ingredients = pd.DataFrame({
    'Ingredient': ingredients,
    'Cleaned_Ingredient': cleaned_ingredients,
    'Cluster': cluster_labels
})
clustered_ingredients.to_csv('data/clustered_ingredients.csv', index=False)

### Example usage to load embeddings and perform a simple query

In [ ]:
ingredient_embeddings = np.load('embeddings/ingredient_embeddings.npy')

# Load the clustered data (if needed)
clustered_ingredients = pd.read_csv('data/clustered_ingredients.csv')

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import random
import torch

# Load the pre-trained Sentence Transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Create a FAISS index for fast similarity search
dimension = ingredient_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)  # L2 distance (Euclidean)
index.add(ingredient_embeddings)  # Add your embeddings to the index

# User query
query = input("Enter the product name for which you want recommendations for: ")
query_embedding = model.encode([query])  # Generate embedding for the query

# Find the 5 most similar ingredients
k = 5  # Number of nearest neighbors
distances, indices = index.search(query_embedding, k)

# Retrieve the similar ingredients
similar_ingredients = [data.iloc[i]['ingredients_text'] for i in indices[0] if 0 <= i < len(data)]
#print(f"Ingredients similar to '{query}': {similar_ingredients}")
print(f"Products with similar ingredients:\n {data.loc[indices[0], ['product_name', 'additives_en', 'allergens_en']]}")